# Workforce Dashboard Workflow — DaimonUpDown Workforce Project

## Purpose

This notebook documents the complete workflow used to prepare, validate, visualize, troubleshoot, and publish the Workforce Dashboard.

The workflow uses **MariaDB/SQL, the terminal, CSV data, Tableau Public, and Git/GitHub**.

The Workforce Dashboard focuses on **assignment-level workforce analysis**. Employee retention and turnover are handled separately in the Employee Retention Dashboard.


## 1. Workforce Dashboard Process

The overall workflow was:

**MariaDB → SQL validation → Workforce dataset creation → CSV verification → Tableau Public → Dashboard troubleshooting → Git/GitHub → Publish**

The process was performed from the project terminal.

The Workforce Dashboard is based on assignment-level records rather than employee-level retention records.

## 2. Workforce Dashboard Goal

The dashboard was designed to answer four main questions:

1. How many distinct employees are represented in the assignment data?
2. Which positions have the most assignments?
3. What are the assignment statuses?
4. How do assignment outcomes change over time?

These are descriptive workforce and assignment questions.

They are not employee-retention or employee-turnover calculations.

## 3. Source Data and Grain

The Tableau-ready workforce dataset is:

`dashboard/data/workforce_dashboard.csv`

The verified dataset contains:

- **2,341 rows**
- **40 columns**
- Expected grain: **one row per assignment**

Because the dataset is assignment-level, assignment counts and assignment outcomes should not automatically be interpreted as employee retention or turnover.

## 4. Verify the MariaDB Source

Before creating the dashboard dataset, MariaDB was checked to confirm that the database was available and that the relevant assignment and employee tables existed.

In [ ]:
sudo systemctl status mariadb

sudo mariadb -e "
USE daimonupdown;
SHOW TABLES;
"

If MariaDB is inactive, start the service before running SQL queries.

In [ ]:
sudo systemctl start mariadb

## 5. Validate Assignment Row Count

The assignment table was checked to confirm the expected number of assignment records.

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT COUNT(*) AS assignments
FROM assignments;
"

Verified assignment count: **2,341**.

## 6. Validate Distinct Employees

Because multiple assignments can belong to the same employee, the number of employees must be calculated with `COUNT(DISTINCT employee_id)` rather than a simple row count.

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT COUNT(DISTINCT employee_id) AS distinct_employees
FROM assignments;
"

The Tableau worksheet later verified **1,500 distinct employees** represented in the assignment dataset.

## 7. Check Assignment Statuses

Assignment status values were inspected before building the Tableau status visualization.

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT
    assignment_status,
    COUNT(*) AS assignment_count
FROM assignments
GROUP BY assignment_status
ORDER BY assignment_count DESC;
"

Statuses observed during dashboard development included:

- Active
- Completed
- Ended
- Renewed
- Transferred

## 8. Check Assignment Positions

Assignment positions were inspected so the Tableau position analysis could be validated against the source data.

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT
    position,
    COUNT(*) AS assignment_count
FROM assignments
GROUP BY position
ORDER BY assignment_count DESC;
"

This query supports the Tableau worksheet that shows which positions contain the most assignment records.

## 9. Check Assignment Relationships

Assignment records were checked for employees that do not exist in the employee table.

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT COUNT(*) AS orphaned_assignments
FROM assignments a
LEFT JOIN employees e
    ON a.employee_id = e.employee_id
WHERE e.employee_id IS NULL;
"

This check protects the relationship between assignments and employees.

The project data-quality validation found no unexpected orphan assignment records in the checked relationships.

## 10. Check Assignment Dates

Assignment start and end dates were checked for impossible date relationships.

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT COUNT(*) AS invalid_assignment_dates
FROM assignments
WHERE assignment_end_date IS NOT NULL
  AND assignment_start_date > assignment_end_date;
"

This check identifies records where an assignment ends before it starts.

## 11. Create the Tableau-Ready Workforce Dataset

The project contains the reusable script:

`python/create_workforce_dashboard.py`

The script creates the Tableau-ready workforce dataset from the project data.

In [ ]:
python python/create_workforce_dashboard.py

The script successfully created:

`dashboard/data/workforce_dashboard.csv`

Verified output:

- Rows: **2,341**
- Columns: **40**
- Grain: **one row per assignment**

## 12. Verify the CSV from the Terminal

The generated file was checked from the terminal before being used in Tableau.

In [ ]:
ls -lh dashboard/data/workforce_dashboard.csv

head -n 2 dashboard/data/workforce_dashboard.csv

The file was verified as the Tableau-ready workforce dataset.

## 13. Tableau Worksheet: Assigned Employees

### Purpose

Show the number of distinct employees represented in the assignment dataset.

### Tableau configuration

- Marks → Text: `Employee Id`
- `Employee Id` → Measure → Count (Distinct)

### Verified result

**1,500 distinct employees**.

## 14. Tableau Worksheet: Assignments by Position

### Purpose

Show how assignment records are distributed across workforce positions.

### Tableau configuration

- Columns: `Position`
- Rows: `Assignment Id`
- `Assignment Id` → Measure → Count (Distinct)
- Marks: Bar

This view answers which positions have the largest number of assignment records.

## 15. Tableau Worksheet: Assignments by Status

### Purpose

Show the distribution of assignment records by assignment status.

### Tableau configuration

- Columns: `Assignment Status`
- Rows: `Assignment Id`
- `Assignment Id` → Measure → Count (Distinct)
- Marks: Bar

The statuses observed during development included Active, Completed, Ended, Renewed, and Transferred.

## 16. Tableau Worksheet: Assignment Outcomes Over Time

### Purpose

Show how assignment outcomes change across years.

The view was adjusted to use assignment end dates when analyzing assignment outcomes.

### Reporting-period filter

The source data contained dates extending into 2027 and 2028. The Tableau view was filtered so the displayed analysis ends in **2026**.

### Interpretation

This chart describes assignment outcomes by year. It does not by itself measure employee retention or turnover.

## 17. Tableau Dashboard Construction

The individual worksheets were combined into the Workforce Dashboard.

The dashboard brings together:

**Assigned Employees → Assignments by Position → Assignments by Status → Assignment Outcomes Over Time**

A worksheet is an individual chart or KPI. A dashboard is the presentation layer that combines the worksheets.

## 18. Tableau Troubleshooting

### Worksheet vs Dashboard

A worksheet is an individual visualization. A dashboard combines existing worksheets.

Creating a chart does not automatically create a dashboard.

### Missing worksheet

During development, an expected worksheet was not visible where expected. The worksheet was recreated rather than assuming it was still available.

### Incorrect fields in the time-series view

`Assignment Status` and `Position` were accidentally included in the time-series analysis. They were removed so the view could focus on assignment dates and assignment outcomes.

### Future years appearing

The assignment data contained future dates extending into 2027 and 2028. The Tableau date filter was adjusted so the displayed analysis ends at 2026.

### Assignment-level vs employee-level confusion

Assignment counts cannot automatically be interpreted as employee counts. Distinct employees must be calculated using `COUNT(DISTINCT employee_id)` or Tableau's Count Distinct measure.

## 19. Git and GitHub Workflow

The project repository is connected to GitHub through the `origin` remote.

The normal workflow is:

1. Make or update project files.
2. Check the working tree.
3. Add the intended files.
4. Commit with a descriptive message.
5. Push to `origin/main`.
6. Confirm the branch is synchronized and the working tree is clean.

In [ ]:
git status
git remote -v
git add notebooks/03_workforce_dashboard.ipynb
git commit -m "Document workforce dashboard workflow"
git push
git status

The repository was successfully synchronized with GitHub during the project.

## 20. Final Validation

Before considering the dashboard complete, the following were verified:

- MariaDB source data was available.
- Assignment count was verified at **2,341**.
- Distinct employees represented were verified at **1,500**.
- Assignment statuses were inspected.
- Assignment positions were inspected.
- Assignment relationships were checked.
- Assignment dates were checked.
- The Tableau-ready CSV contained **2,341 rows and 40 columns**.
- Tableau worksheets were created and combined into the dashboard.
- The time-series analysis was limited to the intended reporting period ending in **2026**.

## 21. Final Workforce Dashboard Scope

The Workforce Dashboard tells this story:

**workforce representation → positions → assignment statuses → assignment outcomes over time.**

It is an operational workforce and assignment dashboard.

Employee retention and turnover are handled separately because those questions require employee-level employment and offboarding data.

## 22. Final Status

The Workforce Dashboard was published to Tableau Public.

The Tableau workbook can still be edited and republished later if changes are needed.

The next dashboard in the project is the Employee Retention Dashboard.